# Notebook 03: The DeepHAM policy objective, from scratch

**Course:** Summer School on AI for Economics and Finance · ESOMAS, University of Torino (August 24–26, 2026)
**Session:** Day 2, 15:30 – 17:00: Practical Session on DeepHAM and Continuous-Time Models
**Slides:** `../../slides/DeepHAM_Lecture_slides.pdf`
**Notebook role:** exercise (blanks; solutions in notebook 04)
**Author:** Yucheng Yang (University of Zurich and Swiss Finance Institute). [Course repository](https://github.com/yangycpku/summer-school-AI-for-economics-and-finance-2026)

---


Everything except the policy objective is imported from the modules you already ran in
notebooks 01 and 02. What you write here is the part that encodes the *economics*: the
household budget constraint, competitive factor prices, and the equilibrium concept.

### What you will fill in

`KSPolicyTrainer.loss` unrolls the economy for `t_unroll` periods and returns the negative
mean discounted utility of agent 0. Five blocks are blanked out:

1. **Policy output** — read the consumption share off the policy network.
2. **Equilibrium concept** — `stop_gradient` on the other agents, so this is a Nash
   equilibrium of a game rather than a planner's optimum.
3. **Factor prices** — the rental rate $R$ and wage $w$ from a Cobb–Douglas technology.
4. **Budget constraint** — cash on hand, consumption, and next-period capital.
5. **Objective** — accumulate $\beta^{t}\log c_t$.

The terminal bootstrap at `t == t_unroll - 1` is given to you: it is where the value
networks from step 4 enter.

> **Note.** The blanks are literal `...` (Python's `Ellipsis`), so the class *definition*
> and the `KSPolicyTrainer(...)` constructor both succeed. The failure comes later, when
> `ptrainer.train(...)` asks TensorFlow to trace `loss`: you will see a `TypeError` about
> an `ellipsis` operand, raised at the first line that consumes a blank you have not
> filled. Work down the TODOs in order and that error walks forward with you.

Run this at `RUN_MODE = "smoke"` first: it exercises the whole loop in a few minutes and
tells you whether your objective is even well-formed. Only then move up.

In [ ]:
RUN_MODE = "smoke"   # one of: "smoke", "teaching", "production"
SEED_INDEX = 3       # which entry of config["random_seed"] to use

## 1. Move into the code directory

DeepHAM is a package of plain Python modules (`param.py`, `dataset.py`, `value.py`,
`policy.py`, ...) that expect to be imported with `src/` as the working directory, with the
data alongside it in `../data`.

On Nuvolos the notebook server starts in `/files`, which mirrors the course repository, so
the path below is the same one you see on GitHub. If you are running from a local clone or
from Colab instead, just open the notebook from inside `DeepHAM_nuvolos/src` and the cell
leaves the working directory alone.

In [ ]:
import os
import sys

# On Nuvolos the kernel starts in /files, which mirrors the course repository.
NUVOLOS_SRC = "/files/day1/Yang/code/DeepHAM_nuvolos/src"
if os.path.isdir(NUVOLOS_SRC):
    os.chdir(NUVOLOS_SRC)

# Anywhere else (local clone, Colab) the notebook's own folder is already src/.
if not os.path.isfile("param.py"):
    raise FileNotFoundError(
        f"Expected to be in DeepHAM_nuvolos/src, but the working directory is "
        f"{os.getcwd()!r}. On Nuvolos that is {NUVOLOS_SRC!r}; elsewhere, open this "
        f"notebook from inside src/ or os.chdir() there yourself."
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())

### Imports

Note what is **not** imported: `KSPolicyTrainer`. That is the class you are about to write.
Everything it needs from `policy.py` — the base class and the module-level constants
`DTYPE`, `NP_DTYPE`, `EPSILON` — is imported explicitly, because the class body below is
executed in the notebook rather than inside `policy.py`.

In [ ]:
import json
import time
import datetime

import numpy as np
import tensorflow as tf

import simulation_KS as KS
from param import KSParam
from dataset import KSInitDataSet
from value import ValueTrainer
from simulation_KS import simul_shocks, simul_k
from util import print_elapsedtime
from util import set_random_seed

# The subclass defined below lives in this notebook rather than in policy.py, so it
# needs the base class *and* the module-level constants that policy.py defines.
from policy import PolicyTrainer, DTYPE, NP_DTYPE, EPSILON

## 2. Choose the run mode

Everything expensive in DeepHAM is controlled by a handful of numbers. The cell below maps
`RUN_MODE` onto them, so the notebook can be run end to end in a few minutes during class
and at the published setting afterwards.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| policy gradient steps | 100 | 1,500 | 10,000 |
| unroll horizon $T$ | 60 | 150 | 150 |
| simulated paths | 64 | 192 | 384 |
| value-net epochs | 10 | 60 | 200 |
| measured wall clock | 2.2 min | 15.6 min | ~90 min |
| mean capital reached | ~11 | ~32 | ~39 (the KS level) |

Timings are measured on a Colab A100; a Nuvolos CPU is slower, though most of the wall
clock is the NumPy simulation rather than the networks. The capital row is the honest
summary of what each budget buys: `smoke` exercises every code path but converges to
nothing, `teaching` gets most of the way to the Krusell–Smith level of $K pprox 39$, and
`production` reproduces the published results (the reference run shipped in
`../data/simul_results` took 5,278 s). Two invariants are asserted rather than
left implicit, because violating either fails deep inside the training loop with an opaque
shape error:

* `policy_config["valid_size"] == dataset_config["n_path"]` — the fixed validation batch is
  built from `init_ds.datadict`, while its shocks are simulated with `valid_size` rows.
* `policy_config["batch_size"] <= policy_config["valid_size"]` — asserted inside
  `PolicyTrainer.train`.

In [ ]:
CONFIG_PATH = "./configs/KS/game_nn_n50_0fm1gm.json"   # 0 fixed moments, 1 learned generalized moment
EXP_NAME = "1gm_exercise"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

seed = config["random_seed"][SEED_INDEX]
set_random_seed(seed)
print(f"Solving {CONFIG_PATH} with seed {seed} (index {SEED_INDEX})")
print(
    f'n_fm = {config["n_fm"]} fixed moment(s), '
    f'n_gm = {config["n_gm"]} generalized moment(s), '
    f'{config["n_agt"]} agents'
)

In [ ]:
# Training budget, dispatched on RUN_MODE (see the run-mode cell above).
if RUN_MODE == "smoke":            # exercises every code path, converges to nothing much
    N_PATH, T_BURN = 64, 300
    V_T, V_COUNT, V_EPOCH = 700, 400, 10
    NUM_STEP, T_UNROLL = 100, 60
    FREQ_VALID, FREQ_UPDATE_V = 50, 50
elif RUN_MODE == "teaching":       # gets most of the way to the KS capital stock
    N_PATH, T_BURN = 192, 2000
    V_T, V_COUNT, V_EPOCH = 1200, 600, 60
    NUM_STEP, T_UNROLL = 1500, 150
    FREQ_VALID, FREQ_UPDATE_V = 250, 500
elif RUN_MODE == "production":     # the setting behind the published results
    N_PATH, T_BURN = 384, 6000
    V_T, V_COUNT, V_EPOCH = 2000, 800, 200
    NUM_STEP, T_UNROLL = 10000, 150
    FREQ_VALID, FREQ_UPDATE_V = 500, 2000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

config["dataset_config"]["n_path"] = N_PATH
config["dataset_config"]["t_burn"] = T_BURN
config["value_config"]["T"] = V_T
config["value_config"]["t_count"] = V_COUNT
config["value_config"]["num_epoch"] = V_EPOCH
config["policy_config"]["num_step"] = NUM_STEP
config["policy_config"]["t_unroll"] = T_UNROLL
config["policy_config"]["freq_valid"] = FREQ_VALID
config["policy_config"]["freq_update_v"] = FREQ_UPDATE_V
# valid_size must match n_path: the validation batch comes from init_ds.datadict
# (n_path rows) while its shocks are simulated with valid_size rows.
config["policy_config"]["valid_size"] = N_PATH
config["policy_config"]["batch_size"] = min(config["policy_config"]["batch_size"], N_PATH)
config["value_config"]["batch_size"] = min(config["value_config"]["batch_size"], N_PATH)

assert config["policy_config"]["valid_size"] == config["dataset_config"]["n_path"]
assert config["policy_config"]["batch_size"] <= config["policy_config"]["valid_size"]
assert config["value_config"]["t_count"] < config["value_config"]["T"] - 1, \
    "t_count must leave at least one time slice inside the value simulation"
assert config["policy_config"]["num_step"] >= config["policy_config"]["freq_valid"], \
    "num_step < freq_valid gives zero training epochs (n_epoch = num_step // freq_valid)"

print(
    f"RUN_MODE={RUN_MODE}: {NUM_STEP} policy steps, unroll {T_UNROLL}, "
    f"{N_PATH} paths, {V_EPOCH} value-net epochs"
)

### Where the results go

In [ ]:
mparam = KSParam(config["n_agt"], config["beta"], config["mats_path"])

# The run mode is part of the directory name, so a quick smoke run can never overwrite a
# long production run -- and neither can overwrite the reference solutions shipped in the
# repository (game_nn_n50_1fm1, game_nn_n50_1gm3).
model_path = "../data/simul_results/KS/game_{}_n{}_{}_{}".format(
    config["dataset_config"]["value_sampling"], config["n_agt"], EXP_NAME, RUN_MODE
)
config["model_path"] = model_path
config["current_time"] = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
os.makedirs(model_path, exist_ok=True)
with open(os.path.join(model_path, "config_beg.json"), "w") as f:
    json.dump(config, f)
print("Results will be written to", model_path)

## 3. Build the initial dataset

`KSInitDataSet` first burns in a panel of `n_path` economies of `n_agt` agents under the
Krusell–Smith benchmark policy, so that the cross-section of wealth starts from its
ergodic distribution. The value networks are then trained on data simulated from an
*initial* policy: either that same benchmark policy (`init_with_bchmk = true`) or a
constant consumption share (`init_with_bchmk = false`, the setting used here).

In [ ]:
start_time = time.monotonic()

init_ds = KSInitDataSet(mparam, config)
value_config = config["value_config"]

if config["init_with_bchmk"]:
    init_policy = init_ds.k_policy_bchmk        # the KS benchmark (b-spline) policy
    policy_type = "pde"
else:
    init_policy = init_ds.c_policy_const_share  # a constant consumption share
    policy_type = "nn_share"

# Supervised targets for the value nets: discounted utility along simulated paths.
train_vds, valid_vds = init_ds.get_valuedataset(init_policy, policy_type, update_init=False)
print_elapsedtime(time.monotonic() - start_time)

## 4. Pre-train the value networks

`num_vnet` independent value networks are fitted to the same targets. They are used as the
terminal bootstrap $\beta^{T}V(s_T)$ that closes the finite unroll in the policy objective;
averaging several of them reduces the variance of that bootstrap.

In [ ]:
vtrainers = []
for i in range(value_config["num_vnet"]):
    config["vnet_idx"] = str(i)
    vtrainers.append(ValueTrainer(config))

for i, vtr in enumerate(vtrainers):
    print(f"--- value net {i} ---")
    vtr.train(train_vds, valid_vds, value_config["num_epoch"], value_config["batch_size"])

## 5. Your turn: the policy trainer

`PolicyTrainer` (the base class, imported above) handles state preparation, the policy
network, gradients and the training loop. Read it in `policy.py` before you start — you do
not need to change any of it.

`KSPolicyTrainer` below is the Krusell–Smith specialisation. Fill in the five TODO blocks.

**Reference formulas.** With Cobb–Douglas production $Y_t = Z_t K_t^{\alpha} L_t^{1-\alpha}$
and depreciation $\delta$:

$$R_t = 1 - \delta + \alpha Z_t \left(\frac{K_t}{L_t}\right)^{\alpha-1},
\qquad
w_t = (1-\alpha) Z_t \left(\frac{K_t}{L_t}\right)^{\alpha}.$$

An agent employed this period ($z^i_t = 1$) supplies $\bar{l}$ units of labour and pays the
labour tax $\tau_t$; an unemployed agent receives the benefit $\mu w_t$. So

$$\text{wealth}^i_t = R_t a^i_t + (1-\tau_t) w_t \bar{l}\, z^i_t + \mu w_t (1 - z^i_t),
\qquad a^i_{t+1} = \text{wealth}^i_t - c^i_t.$$

The attributes you need are on `self.mparam`: `alpha`, `delta`, `l_bar`, `mu`, `n_agt`.

In [ ]:
class KSPolicyTrainer(PolicyTrainer):
    def __init__(self, vtrainers, init_ds, policy_path=None):
        super().__init__(vtrainers, init_ds, policy_path)
        if self.config["init_with_bchmk"]:
            init_policy = self.init_ds.k_policy_bchmk
            policy_type = "pde"
        else:
            init_policy = self.init_ds.c_policy_const_share
            policy_type = "nn_share"
        self.policy_ds = self.init_ds.get_policydataset(init_policy, policy_type, update_init=False)

    @tf.function
    def loss(self, input_data):
        k_cross = input_data["k_cross"]
        ashock, ishock = input_data["ashock"], input_data["ishock"]
        util_sum = 0

        for t in range(self.t_unroll):
            k_mean = tf.reduce_mean(k_cross, axis=1, keepdims=True)
            k_mean_tmp = tf.tile(k_mean, [1, self.mparam.n_agt])
            k_mean_tmp = tf.expand_dims(k_mean_tmp, axis=-1)
            i_tmp = ishock[:, :, t:t+1] # n_path*n_agt*1
            a_tmp = tf.tile(ashock[:, t:t+1], [1, self.mparam.n_agt])
            a_tmp = tf.expand_dims(a_tmp, axis=2) # n_path*n_agt*1
            basic_s_tmp = tf.concat([tf.expand_dims(k_cross, axis=-1), k_mean_tmp, a_tmp, i_tmp], axis=-1)
            basic_s_tmp = self.init_ds.normalize_data(basic_s_tmp, key="basic_s", withtf=True)
            full_state_dict = {
                "basic_s": basic_s_tmp,
                "agt_s": self.init_ds.normalize_data(tf.expand_dims(k_cross, axis=-1), key="agt_s", withtf=True)
            }
            if t == self.t_unroll - 1:
                value = 0
                for vtr in self.vtrainers:
                    value += self.init_ds.unnormalize_data(
                        vtr.value_fn(full_state_dict)[..., 0], key="value", withtf=True)
                value /= self.num_vnet
                util_sum += self.discount[t]*value
                continue

            # TODO (1) -- the policy network returns a consumption *share* of wealth,
            # one number per agent. Shape: (batch, n_agt).
            c_share = ...
            if self.policy_config["opt_type"] == "game":
                # TODO (2) -- Nash equilibrium, not a planner's problem: agent 0 optimises
                # while everyone else is held fixed. Detach agents 1..N-1 from the graph
                # so no gradient flows through their choices.
                c_share = ...
            # labor tax rate - depend on ashock
            tau = tf.where(ashock[:, t:t+1] < 1, self.mparam.tau_b, self.mparam.tau_g)
            # total labor supply - depend on ashock
            emp = tf.where(
                ashock[:, t:t+1] < 1,
                self.mparam.l_bar*self.mparam.er_b,
                self.mparam.l_bar*self.mparam.er_g
            )
            tau, emp = tf.cast(tau, DTYPE), tf.cast(emp, DTYPE)
            # TODO (3) -- competitive factor prices from a Cobb-Douglas technology
            # Y = Z K^alpha L^(1-alpha), with capital depreciating at rate delta.
            # `emp` is aggregate labour supply and `k_mean` is aggregate capital.
            R = ...     # gross return on capital, 1 - delta + marginal product
            wage = ...  # marginal product of labour
            # TODO (4) -- the budget constraint. Cash on hand is the return on last
            # period's capital, plus after-tax labour income if employed
            # (ishock == 1, earning l_bar at wage `wage`, taxed at `tau`), plus the
            # unemployment benefit mu*wage if not.
            wealth = ...
            # Consumption is the chosen share of wealth, clipped away from the two
            # boundaries so that log utility and next-period capital stay finite.
            csmp = tf.clip_by_value(c_share * wealth, EPSILON, wealth-EPSILON)
            k_cross = ...   # whatever is not consumed is carried into t+1
            # TODO (5) -- accumulate discounted flow utility. Preferences are log.
            # `self.discount` is precomputed as beta**arange(t_unroll).
            util_sum += ...

        if self.policy_config["opt_type"] == "socialplanner":
            output_dict = {
                "m_util": -tf.reduce_mean(util_sum), 
                "k_end": tf.reduce_mean(k_cross)
                }
        elif self.policy_config["opt_type"] == "game":
            # optimizing agent 0 only
            output_dict = {
                "m_util": -tf.reduce_mean(util_sum[:, 0]),
                "k_end": tf.reduce_mean(k_cross)
                }
        return output_dict

    def update_policydataset(self, update_init=False):
        self.policy_ds = self.init_ds.get_policydataset(self.current_c_policy, "nn_share", update_init)

    def get_valuedataset(self, update_init=False):
        return self.init_ds.get_valuedataset(self.current_c_policy, "nn_share", update_init)

    def current_c_policy(self, k_cross, ashock, ishock):
        k_mean = np.mean(k_cross, axis=1, keepdims=True)
        k_mean = np.repeat(k_mean, self.mparam.n_agt, axis=1)
        ashock = np.repeat(ashock, self.mparam.n_agt, axis=1)
        basic_s = np.stack([k_cross, k_mean, ashock, ishock], axis=-1)
        basic_s = self.init_ds.normalize_data(basic_s, key="basic_s")
        basic_s = basic_s.astype(NP_DTYPE)
        full_state_dict = {
            "basic_s": basic_s,
            "agt_s": self.init_ds.normalize_data(k_cross[:, :, None], key="agt_s")
        }
        c_share = self.policy_fn(full_state_dict)[..., 0]
        return c_share

    def simul_shocks(self, n_sample, T, mparam, state_init):
        return KS.simul_shocks(n_sample, T, mparam, state_init)

## 6. Train

If the next cell raises during tracing, re-read the traceback: it names the blank that is
still `...`.

In [ ]:
policy_config = config["policy_config"]
ptrainer = KSPolicyTrainer(vtrainers, init_ds)
ptrainer.train(policy_config["num_step"], policy_config["batch_size"])

## 7. Save, and check that the run produced something sane

In [ ]:
with open(os.path.join(model_path, "config.json"), "w") as f:
    json.dump(config, f)

for i, vtr in enumerate(vtrainers):
    vtr.save_model(os.path.join(model_path, "value{}.weights.h5".format(i)))
ptrainer.save_model(os.path.join(model_path, "policy.weights.h5"))

elapsed = time.monotonic() - start_time
with open(os.path.join(model_path, "time.txt"), "w") as f:
    f.write(f"{CONFIG_PATH} at RUN_MODE={RUN_MODE} took {elapsed:.2f} seconds.\n")

print_elapsedtime(elapsed)
print("Saved to", model_path)

In [ ]:
# End-to-end check: the saved policy should simulate to a finite, economically plausible
# capital stock. This is a sanity check on the pipeline, not on convergence -- a `smoke`
# run is far too short to be accurate.
for fname in ["policy.weights.h5", "config.json", "stats.json"]:
    assert os.path.exists(os.path.join(model_path, fname)), f"missing {fname} in {model_path}"

_state = init_ds.next_batch(16)
_shocks = simul_shocks(16, 50, mparam, _state)
_sim = simul_k(
    16, 50, mparam, ptrainer.current_c_policy,
    policy_type="nn_share", state_init=_state, shocks=_shocks,
)
_K = _sim["k_cross"].mean()
assert np.isfinite(_K) and 1.0 < _K < 500.0, f"implausible mean capital {_K}"
print(f"Check passed: mean capital over a short simulation = {_K:.2f}")

## Takeaway

The whole method is in that one `loss` function: simulate the economy forward under the
current policy, add up discounted utility, hand the tail to a learned value function, and
differentiate the lot. `stop_gradient` on the other agents is the single line that
separates a competitive equilibrium from a planner's allocation.

Solutions are in notebook 04 — and in `policy.py`, which is where this class actually
lives.